# 00 - Qubit và Statevector

Bài này trả lời 3 câu hỏi nền tảng:

1. Một qubit được mô tả bằng cái gì? (vector trạng thái)
2. "Superposition" thực chất là gì? (tổ hợp tuyến tính có chuẩn hóa)
3. Đo (measurement) làm gì với trạng thái đó? (lấy mẫu theo xác suất $\lvert\text{amplitude}\rvert^2$)

Chọn kernel **Python 3.12 (Shared Venv)** (trỏ tới `~/.venv`) rồi chạy từng ô một.

In [ ]:
import numpy as np

from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector
from qiskit.visualization import plot_bloch_multivector, plot_histogram
from qiskit_aer.primitives import SamplerV2
from IPython.display import display

import qiskit
import qiskit_aer

print("qiskit:", qiskit.__version__, "| qiskit-aer:", qiskit_aer.__version__)


def ket(sv, nd=3):
    """In statevector dưới dạng tổng các ket, bỏ qua amplitude ~ 0."""
    phan = []
    for i, bien_do in enumerate(sv.data):
        if abs(bien_do) < 1e-9:
            continue
        bits = format(i, f"0{sv.num_qubits}b")
        if abs(bien_do.imag) < 1e-9:
            he_so = f"{bien_do.real:+.{nd}f}"
        elif abs(bien_do.real) < 1e-9:
            he_so = f"{bien_do.imag:+.{nd}f}i"
        else:
            he_so = f"({bien_do.real:+.{nd}f}{bien_do.imag:+.{nd}f}i)"
        phan.append(f"{he_so}|{bits}>")
    return "  ".join(phan) if phan else "0"


def probs(sv, nd=3):
    """Xác suất đo được từng kết quả, key là chuỗi bit thường (không phải np.str_)."""
    return {str(k): round(float(v), nd) for k, v in sv.probabilities_dict().items()}

## 1. Qubit và hai trạng thái cơ sở

Bit cổ điển chỉ có 0 hoặc 1. Qubit là một **vector đơn vị trong không gian 2 chiều**:

$$|0\rangle = \begin{pmatrix}1\\0\end{pmatrix}, \qquad
  |1\rangle = \begin{pmatrix}0\\1\end{pmatrix}$$

Ký hiệu $|\cdot\rangle$ gọi là *ket*. Hai vector này trực giao và tạo thành **computational basis**
(cơ sở tính toán, còn gọi là cơ sở Z). Trong Qiskit, mọi qubit luôn bắt đầu ở $|0\rangle$ - không có
lệnh "khởi tạo về 0" nào cả.

In [ ]:
zero = Statevector.from_label("0")
one = Statevector.from_label("1")

print("|0> =", zero.data)
print("|1> =", one.data)
print("số qubit:", zero.num_qubits, "| số chiều:", zero.dim)
print("dạng ket :", ket(zero), "và", ket(one))

## 2. Superposition: tổ hợp tuyến tính có chuẩn hóa

Trạng thái tổng quát của 1 qubit:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle,
  \qquad \alpha, \beta \in \mathbb{C}, \qquad |\alpha|^2 + |\beta|^2 = 1$$

- $\alpha, \beta$ gọi là **amplitude** (biên độ xác suất) - là số **phức**.
- Khi đo trong cơ sở Z: $P(0) = |\alpha|^2$, $P(1) = |\beta|^2$ (quy tắc Born).
- Điều kiện $|\alpha|^2 + |\beta|^2 = 1$ là **chuẩn hóa**: tổng xác suất phải bằng 1.

Ba trạng thái hay gặp:

$$|+\rangle = \tfrac{1}{\sqrt2}(|0\rangle + |1\rangle), \qquad
  |-\rangle = \tfrac{1}{\sqrt2}(|0\rangle - |1\rangle), \qquad
  |i\rangle = \tfrac{1}{\sqrt2}(|0\rangle + i|1\rangle)$$

Cả ba đều cho $P(0) = P(1) = 0.5$, nhưng chúng **không phải là cùng một trạng thái**
(mục 6 sẽ chỉ cách phân biệt).

In [ ]:
r2 = 1 / np.sqrt(2)

plus = Statevector([r2, r2])
minus = Statevector([r2, -r2])
i_state = Statevector([r2, 1j * r2])

for ten, sv in [("|+>", plus), ("|->", minus), ("|i>", i_state)]:
    print(f"{ten}: {ket(sv)}   chuẩn hóa? {sv.is_valid()}   xác suất {probs(sv)}")

# Vector chưa chuẩn hóa thì không phải trạng thái hợp lệ
print("\n[1, 1] có hợp lệ không?", Statevector([1, 1]).is_valid())

## 3. Từ mạch ra trạng thái

Cách thường dùng nhất: dựng một `QuantumCircuit`, rồi hỏi Qiskit "chạy mạch này từ
$|0\rangle$ thì ra trạng thái gì". Cổng **Hadamard** $H$ chính là cổng tạo superposition:

$$H|0\rangle = |+\rangle, \qquad H|1\rangle = |-\rangle$$

Lưu ý: `Statevector(qc)` chỉ hoạt động khi mạch **không có lệnh `measure`**, vì đo làm mất
thông tin amplitude.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)

print(qc.draw())

sv = Statevector(qc)  # mặc định chạy từ |0>
print("H|0> =", ket(sv))
print("xác suất:", probs(sv))

# Muốn bắt đầu từ trạng thái khác thì dùng .evolve()
print("H|1> =", ket(one.evolve(qc)))

## 4. Bloch sphere - hình dung trạng thái 1 qubit

Bỏ qua global phase, mọi trạng thái 1 qubit viết được thành:

$$|\psi\rangle = \cos\tfrac{\theta}{2}\,|0\rangle + e^{i\varphi}\sin\tfrac{\theta}{2}\,|1\rangle$$

Hai góc $(\theta, \varphi)$ là tọa độ trên một mặt cầu - **Bloch sphere**:

| Trạng thái | Vị trí trên cầu |
|---|---|
| $\lvert 0\rangle$ | cực bắc, $+Z$ |
| $\lvert 1\rangle$ | cực nam, $-Z$ |
| $\lvert +\rangle$ | $+X$ |
| $\lvert -\rangle$ | $-X$ |
| $\lvert i\rangle$ | $+Y$ |

Điểm quan trọng: **hai trạng thái trực giao** ($|0\rangle$ và $|1\rangle$) nằm **đối diện** nhau
trên cầu, chứ không vuông góc - đó là đặc điểm của biểu diễn Bloch.

In [ ]:
for ten, sv in [("|0>", zero), ("|1>", one), ("|+>", plus), ("|->", minus), ("|i>", i_state)]:
    print(ten)
    display(plot_bloch_multivector(sv))

## 5. Đo (measurement)

Đo trong cơ sở Z trả về 0 hoặc 1 với xác suất $|\alpha|^2$, $|\beta|^2$, và trạng thái
**collapse** về đúng ket vừa đo được. Vì thế:

- **Một lần đo không cho biết $\alpha, \beta$.** Muốn ước lượng xác suất thì phải chạy lại
  mạch nhiều lần - mỗi lần gọi là một **shot**.
- `shots=1000` nghĩa là: chuẩn bị lại từ đầu + đo, lặp 1000 lần.

`SamplerV2` là primitive chuẩn từ Qiskit 1.0 trở đi để lấy kết quả dạng đếm (counts).

In [ ]:
qc_do = QuantumCircuit(1, 1)  # 1 qubit + 1 bit cổ điển
qc_do.h(0)
qc_do.measure(0, 0)  # đo qubit 0 -> bit cổ điển 0
print(qc_do.draw())

shots = 1000
ket_qua = SamplerV2().run([qc_do], shots=shots).result()[0]
counts = ket_qua.data.c.get_counts()  # "c" = tên mặc định của thanh ghi cổ điển

print("\ncounts     :", counts)
print("tần suất   :", {k: round(v / shots, 3) for k, v in counts.items()})
print("lý thuyết  :", probs(sv))
display(plot_histogram(counts))

## 6. Relative phase và cách phân biệt $|+\rangle$ với $|-\rangle$

$|+\rangle$ và $|-\rangle$ chỉ khác nhau dấu của amplitude thứ hai - gọi là **relative phase**.
Đo trong cơ sở Z thì cả hai đều cho 50/50, không phân biệt được. Muốn thấy khác biệt, phải
**đổi cơ sở đo**: thêm $H$ ngay trước khi đo (vì $H|+\rangle = |0\rangle$, $H|-\rangle = |1\rangle$).

Phân biệt với **global phase**: nhân cả trạng thái với $e^{i\gamma}$ (ví dụ $-|+\rangle$) thì
**không thể đo được bằng bất kỳ phép đo nào** - hai vector đó là cùng một trạng thái vật lý.

In [ ]:
print("|+>:", ket(plus), "-> đo cơ sở Z:", probs(plus))
print("|->:", ket(minus), "-> đo cơ sở Z:", probs(minus))
print("-> giống hệt nhau trong cơ sở Z\n")

for ten in ["|+>", "|->"]:
    qc2 = QuantumCircuit(1, 1)
    qc2.h(0)  # |0> -> |+>
    if ten == "|->":
        qc2.z(0)  # |+> -> |->
    qc2.h(0)  # đổi cơ sở: X-basis -> Z-basis
    qc2.measure(0, 0)
    dem = SamplerV2().run([qc2], shots=1000).result()[0].data.c.get_counts()
    print(f"{ten} đo trong cơ sở X: {dem}")

# Global phase thì không quan sát được
print("\n-|+> có xác suất:", probs(Statevector(-plus.data)))

## 7. Nhiều qubit và thứ tự bit (little-endian)

$n$ qubit có $2^n$ amplitude. Qiskit đánh số bit theo kiểu **little-endian**:
**qubit 0 là ký tự bên PHẢI cùng** trong chuỗi kết quả.

Ví dụ 3 qubit, chỉ lật qubit 0 thì kết quả đọc là `001`, không phải `100`.
Đây là lỗi hay gặp nhất khi mới học Qiskit.

In [ ]:
qc3 = QuantumCircuit(3)
qc3.x(0)  # chỉ lật qubit 0
print(qc3.draw())
print("trạng thái:", ket(Statevector(qc3)))

qc3.measure_all()  # thêm barrier + thanh ghi tên "meas"
print("counts:", SamplerV2().run([qc3], shots=100).result()[0].data.meas.get_counts())

## Tóm tắt

| Khái niệm | Ý nghĩa | Qiskit |
|---|---|---|
| Ket $\lvert 0\rangle, \lvert 1\rangle$ | computational basis | `Statevector.from_label("0")` |
| Amplitude $\alpha,\beta$ | số phức, $\lvert\alpha\rvert^2+\lvert\beta\rvert^2=1$ | `sv.data`, `sv.is_valid()` |
| Xác suất | $P(k)=\lvert\text{amplitude}_k\rvert^2$ | `sv.probabilities_dict()` |
| Superposition | tổ hợp tuyến tính các ket | `qc.h(0)` |
| Đo | lấy mẫu + collapse | `qc.measure(...)` + `SamplerV2` |
| Relative phase | quan sát được (đổi cơ sở đo) | thêm `h` trước `measure` |
| Global phase | không quan sát được | - |
| Thứ tự bit | little-endian, qubit 0 bên phải | `'001'` = qubit 0 đang 1 |

### Thử tự làm

1. Đổi `qc.h(0)` thành `qc.ry(np.pi/3, 0)` và kiểm tra counts có khớp `probs()` không.
2. Tạo trạng thái $\frac{1}{\sqrt3}|0\rangle + \sqrt{\frac{2}{3}}|1\rangle$ bằng `Statevector`
   và xác nhận `is_valid()`.
3. Vẽ Bloch sphere của $\frac{1}{\sqrt2}(|0\rangle - i|1\rangle)$ và đoán xem nó ở đâu trước khi chạy.

Bài tiếp theo: `01_cong_1_qubit.ipynb`.